# Xenium Glioblastoma To AnnData

This notebook assembles one Xenium sample from `/Volumes/processing2/glioblastoma-xenium` into a KaroSpace-ready `.h5ad`.

The core path is:

1. Load experiment metadata and cell-level coordinates.
2. Read the sparse HDF5 cell-feature matrix.
3. Keep `Gene Expression` features by default.
4. Build a cell-by-gene sparse `AnnData` with `obsm["spatial"]`.
5. Write and reload the assembled `.h5ad`.

The directory currently contains multiple samples. The notebook is sample-aware and defaults to `SNU33`.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

os.environ.setdefault("KMP_WARNINGS", "0")
os.environ.setdefault("NUMBA_DISABLE_JIT", "1")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mplconfig"))

import anndata as ad
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse

ad.settings.allow_write_nullable_strings = True

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 60)

PROJECT_ROOT = Path("/Users/chrislangseth/work/karolinska_institutet/projects/KaroSpaceDataWrangling")
ROOT = PROJECT_ROOT
DATA_ROOT = Path("/Volumes/processing2/glioblastoma-xenium")
SAMPLE_ID = "SNU33"
DATA_DIR = DATA_ROOT / SAMPLE_ID

CELLS_PATH = DATA_DIR / "cells.csv.gz"
MATRIX_H5_PATH = DATA_DIR / "cell_feature_matrix.h5"
EXPERIMENT_PATH = DATA_DIR / "experiment.xenium"
METRICS_PATH = DATA_DIR / "metrics_summary.csv"
MAJOR_ANNOTATION_PATH = DATA_DIR / "summarymajor_celltype_annotation.csv"
MINOR_ANNOTATION_PATH = DATA_DIR / "summaryminor_celltype_annotation.csv"

OUTPUT_DIR = ROOT / "data" / "processed" / "glioblastoma-xenium"
OUTPUT_PATH = OUTPUT_DIR / f"{SAMPLE_ID.lower()}.h5ad"

INCLUDE_CONTROL_FEATURES = False
WRITE_OUTPUT = False
RELOAD_OUTPUT = True

available_samples = sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir() and not p.name.startswith("._"))
ROOT, DATA_ROOT.exists(), SAMPLE_ID, DATA_DIR.exists(), available_samples, OUTPUT_PATH


## Input inspection

The cell table and experiment metadata provide the per-cell coordinates and run context. The matrix itself is already sparse inside the HDF5 file.


In [ ]:
cells = pd.read_csv(CELLS_PATH)
experiment = json.load(open(EXPERIMENT_PATH))
metrics = pd.read_csv(METRICS_PATH)

print("sample:", SAMPLE_ID)
print("cells:", cells.shape)
print("experiment run_name:", experiment.get("run_name"))
print("experiment region_name:", experiment.get("region_name"))
print("matrix path exists:", MATRIX_H5_PATH.exists())

cells.head()


In [ ]:
with h5py.File(MATRIX_H5_PATH, "r") as f:
    matrix_shape = tuple(int(x) for x in f["matrix/shape"][:])
    barcodes_preview = [x.decode() for x in f["matrix/barcodes"][:5]]
    feature_names_preview = [x.decode() for x in f["matrix/features/name"][:10]]
    feature_types = sorted(set(x.decode() for x in f["matrix/features/feature_type"][:]))

print("matrix shape [features, cells]:", matrix_shape)
print("feature types:", feature_types)
print("first barcodes:", barcodes_preview)
print("first features:", feature_names_preview)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(cells["transcript_counts"], bins=60, ax=axes[0])
axes[0].set_title("transcript_counts")

sns.histplot(cells["cell_area"], bins=60, ax=axes[1])
axes[1].set_title("cell_area")

axes[2].scatter(
    cells["x_centroid"],
    cells["y_centroid"],
    s=0.2,
    c=cells["transcript_counts"],
    cmap="viridis",
    linewidths=0,
)
axes[2].set_title("Cell centroids")
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()


## Build sparse `AnnData`

The HDF5 matrix is stored in CSC-like 10x format. The loader below builds a sparse cell-by-feature matrix, aligns it to the cell table, and filters out control features unless explicitly requested.


In [ ]:
def sanitize_obs_for_h5ad(obs_df: pd.DataFrame) -> pd.DataFrame:
    out = obs_df.copy()
    for col in out.columns:
        ser = out[col]
        if isinstance(ser.dtype, pd.CategoricalDtype):
            continue
        if pd.api.types.is_object_dtype(ser) or pd.api.types.is_string_dtype(ser):
            out[col] = ser.astype("string")
    return out


def load_experiment_summary(experiment_obj: dict) -> dict:
    keep = [
        "run_name",
        "region_name",
        "preservation_method",
        "num_cells",
        "transcripts_per_cell",
        "transcripts_per_100um",
        "cassette_name",
        "slide_id",
        "panel_name",
        "panel_organism",
        "panel_tissue_type",
        "panel_num_targets_predesigned",
        "panel_num_targets_custom",
        "pixel_size",
        "analysis_sw_version",
    ]
    return {k: experiment_obj.get(k) for k in keep}


def build_obs(cells_df: pd.DataFrame, experiment_obj: dict) -> pd.DataFrame:
    obs = cells_df.copy()
    obs["cell_id"] = obs["cell_id"].astype("string")

    for col in ["transcript_counts", "control_probe_counts", "control_codeword_counts", "unassigned_codeword_counts", "deprecated_codeword_counts", "total_counts"]:
        obs[col] = pd.to_numeric(obs[col], errors="coerce").astype("Int32")

    for col in ["x_centroid", "y_centroid", "cell_area", "nucleus_area"]:
        obs[col] = pd.to_numeric(obs[col], errors="coerce").astype("float32")

    obs["dataset_id"] = pd.Categorical(["glioblastoma-xenium"] * len(obs))
    obs["sample_id"] = pd.Categorical([SAMPLE_ID] * len(obs))
    obs["section_id"] = pd.Categorical([experiment_obj.get("region_name", SAMPLE_ID)] * len(obs))
    obs["run_name"] = pd.Categorical([experiment_obj.get("run_name", "unknown")] * len(obs))
    obs["panel_name"] = pd.Categorical([experiment_obj.get("panel_name", "unknown")] * len(obs))
    obs["preservation_method"] = pd.Categorical([experiment_obj.get("preservation_method", "unknown")] * len(obs))

    obs = sanitize_obs_for_h5ad(obs)
    return obs.set_index("cell_id", drop=True)


def load_sparse_matrix_and_var(path: Path, include_control_features: bool = False) -> tuple[sparse.csr_matrix, pd.DataFrame, pd.Index]:
    with h5py.File(path, "r") as f:
        data = f["matrix/data"][:]
        indices = f["matrix/indices"][:]
        indptr = f["matrix/indptr"][:]
        shape = tuple(int(x) for x in f["matrix/shape"][:])

        barcodes = pd.Index([x.decode() for x in f["matrix/barcodes"][:]], name="cell_id")
        feature_ids = np.array([x.decode() for x in f["matrix/features/id"][:]], dtype=object)
        feature_names = np.array([x.decode() for x in f["matrix/features/name"][:]], dtype=object)
        feature_types = np.array([x.decode() for x in f["matrix/features/feature_type"][:]], dtype=object)
        genomes = np.array([x.decode() for x in f["matrix/features/genome"][:]], dtype=object)

    X = sparse.csc_matrix((data, indices, indptr), shape=shape).transpose().tocsr()

    var = pd.DataFrame(
        {
            "feature_id": feature_ids,
            "feature_name": feature_names,
            "feature_type": feature_types,
            "genome": genomes,
        },
        index=pd.Index(feature_names.astype(str), name="gene"),
    )

    if not include_control_features:
        keep = var["feature_type"].eq("Gene Expression").to_numpy()
        X = X[:, keep]
        var = var.loc[keep].copy()

    if not var.index.is_unique:
        seen = {}
        new_index = []
        for gene in var.index.astype(str):
            count = seen.get(gene, 0)
            new_index.append(gene if count == 0 else f"{gene}-{count}")
            seen[gene] = count + 1
        var.index = pd.Index(new_index, name="gene")

    return X, var, barcodes


obs = build_obs(cells, experiment)
X, var, barcodes = load_sparse_matrix_and_var(MATRIX_H5_PATH, include_control_features=INCLUDE_CONTROL_FEATURES)

print("obs shape:", obs.shape)
print("matrix shape:", X.shape)
print("var shape:", var.shape)
print("barcode sets identical:", barcodes.isin(obs.index).all() and obs.index.isin(barcodes).all())


In [ ]:
missing_from_obs = barcodes.difference(obs.index)
missing_from_matrix = obs.index.difference(barcodes)
if len(missing_from_obs) or len(missing_from_matrix):
    raise ValueError(
        f"barcode mismatch: missing_from_obs={len(missing_from_obs)}, missing_from_matrix={len(missing_from_matrix)}"
    )

obs_aligned = obs.reindex(barcodes).copy()
adata = ad.AnnData(X=X, obs=obs_aligned, var=var.copy())
adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy(dtype=np.float32)
adata.uns["experiment"] = load_experiment_summary(experiment)
adata.uns["metrics_summary"] = metrics.iloc[0].to_dict()

adata


In [ ]:
print("obs x vars:", adata.n_obs, adata.n_vars)
print("X format:", type(adata.X), "nnz:", adata.X.nnz)
print("obsm keys:", list(adata.obsm.keys()))
print("feature types kept:", adata.var["feature_type"].value_counts().to_dict())

adata.obs.head()


## Optional annotation tables

The two `summary...celltype_annotation.csv` files are gene-centric reference tables, not per-cell assignments. They are useful for marker interpretation, but they are not merged into `obs`.


In [ ]:
major_annotation = pd.read_csv(MAJOR_ANNOTATION_PATH)
minor_annotation = pd.read_csv(MINOR_ANNOTATION_PATH)

print("major annotation shape:", major_annotation.shape)
print("minor annotation shape:", minor_annotation.shape)

major_annotation[["Genes", "Single cell type expression cluster", "Single cell type specificity"]].head()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    adata.obsm["spatial"][:, 0],
    adata.obsm["spatial"][:, 1],
    s=0.2,
    c=adata.obs["transcript_counts"].astype(float).to_numpy(),
    cmap="viridis",
    linewidths=0,
)
ax.set_title("Spatial coordinates colored by transcript_counts")
ax.set_xlabel("x_centroid")
ax.set_ylabel("y_centroid")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## Write and reload

The expression matrix stays sparse in the written `.h5ad`.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if WRITE_OUTPUT:
    adata.write_h5ad(OUTPUT_PATH)
    print(f"Wrote: {OUTPUT_PATH}")
else:
    print("WRITE_OUTPUT is False")
    print(f"Planned output: {OUTPUT_PATH}")


In [ ]:
if RELOAD_OUTPUT and OUTPUT_PATH.exists():
    adata_reloaded = ad.read_h5ad(OUTPUT_PATH)
    print(adata_reloaded)
    print("reloaded X format:", type(adata_reloaded.X))
else:
    adata_reloaded = None
    print("Reload skipped")


In [ ]:
karospace_summary = {
    "sample_id": SAMPLE_ID,
    "output_path": str(OUTPUT_PATH),
    "section_key": "section_id",
    "sample_key": "sample_id",
    "spatial_key": "obsm['spatial']",
    "feature_filter": "Gene Expression only" if not INCLUDE_CONTROL_FEATURES else "includes control features",
    "candidate_color_columns": [
        "transcript_counts",
        "cell_area",
        "nucleus_area",
        "preservation_method",
        "panel_name",
    ],
}

print(json.dumps(karospace_summary, indent=2))
